# Lab 11: Environment Teardown and Workshop Architecture Recap
Building Enterprise AI Agents on **Gemini Enterprise Agent Platform (GEAP)**: Pulse Telco Workshop

---

**Note for temporary workshop accounts (*devstarNNNN@gcplab.me* / *pulse-telco-ws-NNNN*):** No manual action is required if you are using a temporary pre-provisioned workshop account because the sandbox project and every resource inside it are deleted automatically when the sandbox expires. Run this lab if you want to practice a production-grade reverse-dependency teardown, or if you executed the workshop inside your own persistent GCP project.

## 🎯 Objectives
Provisioning cloud infrastructure is forgiving because APIs happily create dependent resources on demand; tearing them down is where hidden coupling surfaces. If you delete a service account before removing its project-level IAM bindings, Google Cloud leaves orphaned *deleted:serviceAccount:...* tombstones in your IAM policy. If you delete an Agent Runtime instance without passing *force=true*, child sessions, Memory Bank records, and active Code Execution sandboxes block the deletion or linger.

In this final lab you will:
1. Review the complete dependency inventory created across Labs 00 to 08 and understand why teardown must run in **reverse dependency order**.
2. Remove **only the workshop resources while preserving the GCP project itself** using a single idempotent cleanup script that defaults to **dry-run mode**, continues past missing resources (*already gone*), and waits for asynchronous deletions to complete.
3. Run a post-teardown audit and inspect what is intentionally left in place (enabled APIs, audit logs, and platform-managed IAM bindings).

**How the cleanup cell operates:**
* *CONFIRM_RESOURCE_CLEANUP = False* (default): Runs strictly in **read-only dry-run mode**. It discovers active instances, evaluators, and IAM bindings and prints the exact CLI and REST delete commands it would execute without deleting anything.
* *CONFIRM_RESOURCE_CLEANUP = True*: Executes the teardown in reverse dependency order, marks each item as *done*, *already gone*, or *FAILED*, and prints a final summary. ⏱ 5 to 10 minutes, most of which is spent polling until the Agent Runtime instances finish deprovisioning.

**Note on the optional reference production agent:** If you also applied the Terraform configuration in *reference-production-agent/terraform* inside this same project, that deployment shares the Lab 00 BigQuery datasets, persona service accounts, Cloud Run services, and Agent Registry entries. Destroy the Terraform stack first (see its README) or skip this cleanup so you do not pull shared resources out from under it.

### Key references
* 🧹 [Manage deployed agents (delete an Agent Runtime instance)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/manage-deployed-agents)
* 🏛️ [Agent Registry: manage agents](https://docs.cloud.google.com/agent-registry/manage-agents)
* ☁️ [Manage Cloud Run services](https://docs.cloud.google.com/run/docs/managing/services)
* 🔐 [Manage access to projects (remove IAM role bindings)](https://docs.cloud.google.com/iam/docs/granting-changing-revoking-access)

---

## 🗺️ Teardown Sequence (Reverse Dependency Order)

```mermaid
flowchart TD
    S1["1 · Lab 07: online evaluators, eval results bucket"] --> S2["2 · Labs 02/06/08: Agent Runtime instances (force: sessions, memories, sandboxes), wait until gone"]
    S2 --> S3["3 · Labs 06/03: registry entries, Cloud Run services, their images and uploaded sources, private skill (force=true)"]
    S3 --> S4["4 · Lab 05: Model Armor templates (MA_LOCATION + REGION), log-based metric"]
    S4 --> S5["5 · Labs 02 to 08: IAM roles the labs granted (Agent Identities, default compute service account)"]
    S5 --> S6["6 · Lab 00: BigQuery datasets, aspect types, persona bindings and service accounts"]
```

## Step 11.0: Load Workshop Configuration

We load *workshop.env* so the inventory and cleanup scripts target the exact project identifiers, dataset names, and locations configured in earlier labs (*europe-west4* for regional services and *eu* / *EU* for multi-region data, skills, and Model Armor templates).

In [ ]:
# Step 11.0: Load workshop configuration from workshop.env
import os, sys, json, pathlib
os.environ["PATH"] = f"{pathlib.Path.home() / '.local/bin'}:{os.environ.get('PATH', '')}"
os.environ.pop("VIRTUAL_ENV", None)  # prevent parent .venv from leaking into child uv / agents-cli calls
import site, sys
if sys.prefix == sys.base_prefix and os.path.isdir(site.getusersitepackages()):
    site.addsitedir(site.getusersitepackages())
    sys.path.insert(0, site.getusersitepackages())
from dotenv import load_dotenv

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"
assert ENV_FILE.exists(), "workshop.env not found - run Lab00 first"
load_dotenv(ENV_FILE, override=True)

PROJECT_ID            = os.environ["PROJECT_ID"]
PROJECT_NUMBER        = os.environ["PROJECT_NUMBER"]
REGION                = os.environ.get("REGION", "europe-west4")                 # europe-west4: Agent Runtime, Sessions, Memory Bank, Sandbox, Cloud Run, Agent Registry
MODEL_LOCATION        = os.environ.get("MODEL_LOCATION", "eu")                   # eu: multi-region endpoint that serves gemini-3.8-flash and gemini-3.8-live
MODEL                 = os.environ.get("MODEL", "gemini-3.8-flash")              # gemini-3.8-flash
AGENT_NAME            = os.environ.get("AGENT_NAME", "pulse-concierge")          # pulse-concierge
PULSE_SKILLS_LOCATION = os.environ.get("PULSE_SKILLS_LOCATION", "eu")            # eu: Skill Registry multi-region
MA_LOCATION           = os.environ.get("MA_LOCATION", "eu")                      # eu: Model Armor multi-region
BQ_DATASET_CRM        = os.environ.get("BQ_DATASET_CRM", "telco_crm")
BQ_DATASET_BILLING    = os.environ.get("BQ_DATASET_BILLING", "telco_billing")
BQ_DATASET_NETWORK    = os.environ.get("BQ_DATASET_NETWORK", "telco_network")
AGENT_DIR             = REPO_ROOT / AGENT_NAME

# Make every shell (!) and SDK call in this notebook target the workshop project.
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "GOOGLE_API_USE_CLIENT_CERTIFICATE": "false",
    "UV_DEFAULT_INDEX": "https://pypi.org/simple",
    "CLOUDSDK_CORE_PROJECT": PROJECT_ID,
    "CLOUDSDK_CORE_DISABLE_PROMPTS": "1",
})
print(f"Project: {PROJECT_ID} ({PROJECT_NUMBER}) | region: {REGION} | model: {MODEL} @ {MODEL_LOCATION}")
print(f"Skills location: {PULSE_SKILLS_LOCATION} | Model Armor location: {MA_LOCATION}")
print(f"Agent dir: {AGENT_DIR} | BQ datasets: {BQ_DATASET_CRM}, {BQ_DATASET_BILLING}, {BQ_DATASET_NETWORK}")

def terminal(cmd, cwd=None):
    """Print the exact command a cell is about to run, so you can copy/paste it into your own terminal."""
    prefix = f"cd {os.path.relpath(cwd, REPO_ROOT)} && " if cwd else ""
    print(f"$ {prefix}{cmd}\n")

## Step 11.1: Resource Inventory and Reverse-Dependency Teardown Order

| Stage | Created in | Resource | Location | Teardown mechanism |
| :--- | :--- | :--- | :--- | :--- |
| 1 | Lab 07 | Online evaluation monitors attached to workshop agents (if created in Lab 07) | *REGION* | REST *DELETE .../onlineEvaluators/<id>* |
| 1 | Lab 07 | Cloud Storage bucket *gs://<PROJECT_ID>-eval* (managed evaluation artifacts) | *REGION* | *gcloud storage rm -r* |
| 2 | Lab 02 (final deploy in Lab 06) | Agent Runtime instance *pulse-concierge* along with its child Sessions, Memory Bank records, Code Execution sandboxes, and auto-registered Agent Registry entry | *REGION* | REST *DELETE .../reasoningEngines/<id>?force=true* |
| 2 | Lab 08 | Agent Runtime instance *pulse-workflow-agent* (if deployed in Lab 08) | *REGION* | Same REST *DELETE* with *force=true* |
| 3 | Lab 06 | Agent Registry entry and Cloud Run service *pulse-network-ops-agent*, container image package, and staged source archive | *REGION* | Agent Registry, Cloud Run, Artifact Registry, and Cloud Storage delete commands |
| 3 | Lab 03 | Agent Registry entry and Cloud Run service *pulse-iam-context-mcp*, container image package, and staged source archive | *REGION* | Same four delete commands |
| 3 | Lab 03 | Private skill *private-pulse-telco-analytics* and all skill revisions | *eu* and *global* | REST *DELETE* with *force=true* |
| 4 | Lab 05 | Model Armor templates (*pulse-guard-input*, *pulse-guard-output*, and policy aliases) | *eu* and *europe-west4* | REST *DELETE* on regional Model Armor endpoint |
| 4 | Lab 05 | Cloud Logging metric *pulse_blocked_prompts* | Project-wide | *gcloud logging metrics delete* |
| 5 | Labs 02, 03, 05, 08 | Project IAM roles granted by the labs to the agents' Agent Identities (BigQuery, Knowledge Catalog, MCP, Service Usage, Agent Registry, and Model Armor roles) | Project-wide | *gcloud projects remove-iam-policy-binding* |
| 5 | Labs 03, 06 | Project IAM roles granted to the default Compute Engine service account for Cloud Run source builds (*cloudbuild.builds.builder*, *aiplatform.user*) | Project-wide | *gcloud projects remove-iam-policy-binding* |
| 6 | Lab 00 | BigQuery datasets (*telco_crm*, *telco_billing*, *telco_network*), tables, dataset ACLs, and synced Knowledge Catalog entries | *EU* | *bq rm -r -f -d* |
| 6 | Lab 00 | Knowledge Catalog aspect types *telco-governance* and *telco-semantics* | *eu* | *gcloud dataplex aspect-types delete* |
| 6 | Lab 00 | Persona service accounts (*persona-alex-exec*, *persona-elena-care*) and their project-level *bigquery.jobUser* bindings (bindings removed before deleting accounts to prevent orphaned tombstones) | Project-wide | IAM binding removal followed by service account deletion |

### What Is Deliberately Left in Place
* **The GCP project, billing link, and enabled service APIs from Lab 00:** Disabling core APIs can disrupt other workloads in a shared project, and enabled APIs incur zero cost when idle.
* **IAM bindings granted automatically by the Agents CLI:** When *agents-cli deploy --agent-identity* provisions an Agent Identity, it automatically binds core runtime roles (*aiplatform.user*, *browser*, *cloudapiregistry.viewer*, *logging.logWriter*, and *monitoring.metricWriter*, plus *serviceusage.serviceUsageConsumer* which Lab 03 also grants explicitly and Step 11.2 removes). Because the labs did not add those CLI-managed bindings, Step 11.1 lists them for visibility without removing them. Deleting the Agent Runtime instance permanently invalidates the underlying principal, and you can optionally prune any residual binding with:
```bash
gcloud projects remove-iam-policy-binding "$PROJECT_ID" \
  --member="$PULSE_AGENT_PRINCIPAL" --role="<ROLE>" --condition=None
```
* **Shared Cloud Run source infrastructure (*cloud-run-source-deploy* Artifact Registry repository and *run-sources-<PROJECT_ID>-<REGION>* bucket):** Created automatically by *gcloud run deploy --source* and shared across all source deployments in the project. Step 11.2 deletes only the workshop's specific image packages and source archives inside them.
* **Cloud Logging entries:** Audit logs and OpenTelemetry prompt/response events captured in Lab 04 age out automatically according to your log bucket's retention policy (30 days by default for *_Default*).
* **Agent Gateway instances:** Not created by the hands-on notebooks (provisioned only if you ran *reference-production-agent/terraform*). The audit function reports any existing gateways for visibility.
* **Model Armor project floor settings:** Lab 05 presented the floor-setting command as a reference without executing it. If you enabled it manually in your project, disable enforcement with:
```bash
gcloud model-armor floorsettings update \
  --full-uri="projects/$PROJECT_ID/locations/global/floorSetting" \
  --enable-floor-setting-enforcement=FALSE --clear-integrated-services
```
* **Local workspace files (*workshop.env*, *pulse-concierge/*, *pulse-workflow-agent/*, *labs/artifacts/*):** Stored on your local disk; delete the repository directory whenever you are finished inspecting the code.

The code cell below initializes the inventory constants, defines the read-only *audit()* helper, and runs an initial audit so you can inspect every active workshop resource **before** running the teardown.

In [ ]:
# Step 11.1: Define the workshop resource inventory, helper functions, and read-only audit
import subprocess, time, requests

os.environ["GOOGLE_API_USE_CLIENT_CERTIFICATE"] = "false"
REGION                = os.environ.get("REGION", globals().get("REGION", "europe-west4"))
PULSE_SKILLS_LOCATION = os.environ.get("PULSE_SKILLS_LOCATION", "eu")
_ma_template          = os.environ.get("PULSE_MODEL_ARMOR_INPUT_TEMPLATE", "")  # projects/<project>/locations/<location>/templates/<id>
MA_LOCATION           = os.environ.get("MA_LOCATION") or (_ma_template.split("/")[3] if _ma_template.count("/") == 5 else "eu")

AGENT_NAMES      = {AGENT_NAME, "pulse-workflow-agent"}                    # Agent Runtime instances: Lab 02 (+ final deploy in Lab 06), Lab 08
SERVICES         = ["pulse-network-ops-agent", "pulse-iam-context-mcp"]    # Cloud Run service + Agent Registry entry: Lab 06, Lab 03
SKILLS_LOCATION  = PULSE_SKILLS_LOCATION                                   # Lab 03 private skill primary location (eu)
SKILL_LOCATIONS  = list(dict.fromkeys([PULSE_SKILLS_LOCATION, "global"]))  # Check both PULSE_SKILLS_LOCATION (eu) and global
SKILL_ID         = "private-pulse-telco-analytics"
def skill_url(loc=SKILLS_LOCATION):
    return f"https://agentregistry.googleapis.com/v1alpha/projects/{PROJECT_ID}/locations/{loc}/skills/{SKILL_ID}"
SKILL_URL        = skill_url(SKILLS_LOCATION)

MA_LOCATIONS     = list(dict.fromkeys([MA_LOCATION, REGION]))              # Check both MA_LOCATION (eu) and REGION (europe-west4)
def ma_api(loc=MA_LOCATION):
    return f"https://modelarmor.{loc}.rep.googleapis.com/v1/projects/{PROJECT_ID}/locations/{loc}"
MA_API           = ma_api(MA_LOCATION)
MA_TEMPLATES     = ["pulse-guard-input", "pulse-guard-output",             # Lab 05 template names + policy aliases
                    "pulse-concierge-input-policy", "pulse-concierge-output-policy"]
DATASETS         = [BQ_DATASET_CRM, BQ_DATASET_BILLING, BQ_DATASET_NETWORK]  # Lab 00
ASPECT_TYPES     = ["telco-governance", "telco-semantics"]                 # Lab 00
KC_LOCATION      = "eu"                                                     # Knowledge Catalog multi-region used by Lab 00
PERSONA_SAS      = [os.environ.get("PERSONA_ADMIN_SA") or f"persona-alex-exec@{PROJECT_ID}.iam.gserviceaccount.com",
                    os.environ.get("PERSONA_SPECIALIST_SA") or f"persona-elena-care@{PROJECT_ID}.iam.gserviceaccount.com"]
COMPUTE_SA       = f"serviceAccount:{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
COMPUTE_SA_ROLES = {"roles/cloudbuild.builds.builder", "roles/aiplatform.user"}  # Labs 03, 06: use set() if your project relied on them before
LAB_AGENT_ROLES  = {"roles/bigquery.dataViewer", "roles/bigquery.jobUser", "roles/dataplex.catalogViewer", "roles/mcp.toolUser",
                    "roles/serviceusage.serviceUsageConsumer", "roles/agentregistry.viewer", "roles/modelarmor.user"}  # Labs 02, 03, 05, 08
AR_API           = f"https://{REGION}-aiplatform.googleapis.com"

def sh(cmd):
    """Run a read-only command; return its non-empty output lines ([] on error)."""
    return [l.strip() for l in subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.splitlines() if l.strip()]

H = {}
def refresh_token():
    """Access token for the REST calls (valid ~1 h); refreshed at the start of every cell that uses it."""
    H["Authorization"] = "Bearer " + "".join(sh("gcloud auth print-access-token"))

def get_json(url, **params):
    r = requests.get(url, headers=H, params=params)
    return r.json() if r.ok else {}

def workshop_engines():
    """Every Agent Runtime instance named pulse-concierge / pulse-workflow-agent (also copies from repeated deploys), plus the ids in workshop.env."""
    recorded = {os.environ.get(k, "").split("/")[-1] for k in ("PULSE_AGENT_ENGINE", "PULSE_WORKFLOW_AGENT_ENGINE")} - {""}
    engines = get_json(f"{AR_API}/v1/projects/{PROJECT_ID}/locations/{REGION}/reasoningEngines", pageSize=100).get("reasoningEngines", [])
    return [e for e in engines if e.get("displayName") in AGENT_NAMES or e["name"].split("/")[-1] in recorded]

def agent_principals(engines):
    """The workshop agents' Agent Identities: from workshop.env and from the instances that still exist."""
    found = {os.environ.get(k, "") for k in ("PULSE_AGENT_PRINCIPAL", "PULSE_WORKFLOW_AGENT_PRINCIPAL")}
    found |= {"principal://" + e["spec"]["effectiveIdentity"] for e in engines if e.get("spec", {}).get("effectiveIdentity")}
    return sorted(found - {""})

def workshop_evaluators(engines):
    """Lab 07 online evaluators that grade one of the workshop agents."""
    ids = {e["name"].split("/")[-1] for e in engines}
    ids |= {os.environ.get(k, "").split("/")[-1] for k in ("PULSE_AGENT_ENGINE", "PULSE_WORKFLOW_AGENT_ENGINE")} - {""}
    evaluators = get_json(f"{AR_API}/v1beta1/projects/{PROJECT_ID}/locations/{REGION}/onlineEvaluators").get("onlineEvaluators", [])
    return [o for o in evaluators if o.get("agentResource", "").split("/")[-1] in ids or o.get("displayName") == "pulse-quality-monitor"]

def iam_policy():
    return json.loads("".join(sh(f"gcloud projects get-iam-policy {PROJECT_ID} --format=json")) or "{}")

def roles_of(policy, member):
    """Unconditional project roles bound to one member."""
    return {b["role"] for b in policy.get("bindings", []) if member in b.get("members", []) and "condition" not in b}

def audit():
    """Read-only: print every workshop resource that still exists (~1 min)."""
    refresh_token()
    engines, policy = workshop_engines(), iam_policy()
    principals = agent_principals(engines)
    datasets = json.loads("".join(sh(f"bq ls --project_id={PROJECT_ID} --max_results=1000 --format=json")) or "[]")
    lab_roles = [f"agent {p.split('/')[-1]}: {r}" for p in principals for r in sorted(roles_of(policy, p) & LAB_AGENT_ROLES)]
    lab_roles += [f"compute SA: {r}" for r in sorted(roles_of(policy, COMPUTE_SA) & COMPUTE_SA_ROLES)]
    lab_roles += [f"{sa.split('@')[0]}: {r}" for sa in PERSONA_SAS for r in sorted(roles_of(policy, f"serviceAccount:{sa}"))]
    found_skills = [f"{SKILL_ID} ({loc})" for loc in SKILL_LOCATIONS if requests.get(skill_url(loc), headers=H).ok]
    found_templates = [f"{t['name'].split('/')[-1]} ({loc})" for loc in MA_LOCATIONS
                       for t in get_json(f"{ma_api(loc)}/templates").get("templates", []) if t["name"].split("/")[-1] in MA_TEMPLATES]
    checks = {
        "Agent Runtime instances":  [f"{e['displayName']} ({e['name'].split('/')[-1]})" for e in engines],
        "Online evaluators":        [o.get("displayName", o["name"]) for o in workshop_evaluators(engines)],
        "Eval results bucket":      [b for b in sh(f"gcloud storage ls --project={PROJECT_ID}") if b.rstrip("/") == f"gs://{PROJECT_ID}-eval"],
        "Registry agents (auto)":   [a for a in sh(f"gcloud agent-registry agents list --location={REGION} --project={PROJECT_ID} --format='value(displayName)'") if a in AGENT_NAMES],
        "Registry entries":         [s for s in sh(f"gcloud agent-registry services list --location={REGION} --project={PROJECT_ID} --format='value(name.basename())'") if s in SERVICES],
        "Cloud Run services":       [s for s in sh(f"gcloud run services list --region={REGION} --project={PROJECT_ID} --format='value(metadata.name)'") if s in SERVICES],
        "Container images":         [p for p in sh(f"gcloud artifacts packages list --repository=cloud-run-source-deploy --location={REGION} --project={PROJECT_ID} --format='value(name.basename())'") if p in SERVICES],
        "Private skill":            found_skills,
        "Model Armor templates":    found_templates,
        "Log-based metric":         sh(f"gcloud logging metrics list --project={PROJECT_ID} --filter='name=pulse_blocked_prompts' --format='value(name)'"),
        "IAM roles from the labs":  lab_roles,
        "BigQuery datasets":        [d["datasetReference"]["datasetId"] for d in datasets if d["datasetReference"]["datasetId"] in DATASETS],
        "Aspect types":             [a for a in sh(f"gcloud dataplex aspect-types list --location={KC_LOCATION} --project={PROJECT_ID} --format='value(name.basename())'") if a in ASPECT_TYPES],
        "Persona service accounts": [s for s in sh(f"gcloud iam service-accounts list --project={PROJECT_ID} --format='value(email)'") if s in PERSONA_SAS],
    }
    for label, found in checks.items():
        print(f"{label:<26} {', '.join(found) if found else '- none -'}")
    auto = {p.split("/")[-1]: sorted(roles_of(policy, p) - LAB_AGENT_ROLES) for p in principals}
    print("\nFor information (not removed by this lab):")
    print("  roles agents-cli granted itself:", "; ".join(f"agent {k}: {', '.join(v)}" for k, v in auto.items() if v) or "- none -")
    print(f"  Agent Gateways in {REGION}:", ", ".join(sh(f"gcloud network-services agent-gateways list --location={REGION} --project={PROJECT_ID} --format='value(name.basename())'")) or "- none -")

print(f"Agent Runtime + Cloud Run + registry: {REGION} | Model Armor: {', '.join(MA_LOCATIONS)} | skills: {', '.join(SKILL_LOCATIONS)} | Knowledge Catalog: {KC_LOCATION}\n")
audit()

## Step 11.2: Execute Reverse-Dependency Teardown (Dry Run by Default)

Run the cell below as written first: with *CONFIRM_RESOURCE_CLEANUP = False*, it queries your project for active Agent Runtime instances and Agent Identities and prints the exact sequence of CLI and REST deletion commands without modifying any state. Once you have reviewed the plan, change *CONFIRM_RESOURCE_CLEANUP = True* and re-run the cell to execute the teardown. ⏱ 5 to 10 minutes.

Why use the Python teardown runner instead of raw one-liners?
* **Catches duplicate Agent Runtime deployments:** Queries the reasoning engine collection and deletes **every** instance named *pulse-concierge* or *pulse-workflow-agent* (including orphaned instances created if *deployment_metadata.json* was reset), while leaving any *pulse-concierge-prod* instance untouched.
* **Synchronizes on long-running deletions:** Polls the Agent Runtime API until deleted instances return HTTP 404 before removing the IAM bindings and datasets they depended on.
* **Sweeps both regional and multi-region control planes:** Checks both *eu* and *global* when deleting the *private-pulse-telco-analytics* skill (passing *force=true* to purge all revisions), and checks both *eu* and *europe-west4* against the regional Model Armor endpoint when deleting guardrail templates.
* **Diff-aware IAM cleanup:** Reads the project IAM policy once and removes only the lab-granted role bindings that are currently present, avoiding noisy errors for roles that were never bound.
* **Idempotent retry:** If a Knowledge Catalog aspect type reports *FAILED* immediately after dataset deletion because catalog metadata is still synchronizing, wait two minutes and re-run the cell; completed steps report *already gone*.

<details>
<summary><b>Equivalent terminal commands</b> (run from the repository root; the code cell prints the exact commands with your project values)</summary>

```bash
source workshop.env
REGION="${REGION:-europe-west4}"
PULSE_SKILLS_LOCATION="${PULSE_SKILLS_LOCATION:-eu}"
MA_LOCATION="${MA_LOCATION:-eu}"
TOKEN="$(gcloud auth print-access-token)"
# 1. Lab 07: eval results bucket (the cell also deletes online evaluators that grade the workshop agents)
gcloud storage rm -r "gs://${PROJECT_ID}-eval"
# 2. Labs 02/06/08: Agent Runtime instances; force=true also deletes their sessions, memories and sandboxes
for ENGINE in "$PULSE_AGENT_ENGINE" "$PULSE_WORKFLOW_AGENT_ENGINE"; do
  [ -n "$ENGINE" ] && curl -s -X DELETE "https://${REGION}-aiplatform.googleapis.com/v1/${ENGINE}?force=true" -H "Authorization: Bearer ${TOKEN}"
done
# 3. Labs 06/03: registry entries, Cloud Run services, their images and uploaded sources, then the private skill (eu + global, force=true)
for SVC in pulse-network-ops-agent pulse-iam-context-mcp; do
  gcloud agent-registry services delete "$SVC" --location="$REGION" --project="$PROJECT_ID" --quiet
  gcloud run services delete "$SVC" --region="$REGION" --project="$PROJECT_ID" --quiet
  gcloud artifacts packages delete "$SVC" --repository=cloud-run-source-deploy --location="$REGION" --project="$PROJECT_ID" --quiet
  gcloud storage rm "gs://run-sources-${PROJECT_ID}-${REGION}/services/${SVC}/**"
done
for LOC in "$PULSE_SKILLS_LOCATION" global; do
  curl -s -X DELETE "https://agentregistry.googleapis.com/v1alpha/projects/${PROJECT_ID}/locations/${LOC}/skills/private-pulse-telco-analytics?force=true" -H "Authorization: Bearer ${TOKEN}"
done
# 4. Lab 05: Model Armor templates (check both MA_LOCATION and REGION) and the log-based metric
for LOC in "$MA_LOCATION" "$REGION"; do
  for TPL in pulse-guard-input pulse-guard-output pulse-concierge-input-policy pulse-concierge-output-policy; do
    curl -s -X DELETE "https://modelarmor.${LOC}.rep.googleapis.com/v1/projects/${PROJECT_ID}/locations/${LOC}/templates/${TPL}" -H "Authorization: Bearer ${TOKEN}"
  done
done
gcloud logging metrics delete pulse_blocked_prompts --project="$PROJECT_ID" --quiet
# 5. IAM roles the labs granted (errors for roles that were never bound are harmless)
for P in "$PULSE_AGENT_PRINCIPAL" "$PULSE_WORKFLOW_AGENT_PRINCIPAL"; do
  [ -z "$P" ] && continue
  for ROLE in roles/bigquery.dataViewer roles/bigquery.jobUser roles/dataplex.catalogViewer roles/mcp.toolUser \
              roles/serviceusage.serviceUsageConsumer roles/agentregistry.viewer roles/modelarmor.user; do
    gcloud projects remove-iam-policy-binding "$PROJECT_ID" --member="$P" --role="$ROLE" --condition=None --format=none
  done
done
for ROLE in roles/cloudbuild.builds.builder roles/aiplatform.user; do
  gcloud projects remove-iam-policy-binding "$PROJECT_ID" --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" --role="$ROLE" --condition=None --format=none
done
# 6. Lab 00: datasets, aspect types, then the persona service accounts (project binding first)
for DS in "$BQ_DATASET_CRM" "$BQ_DATASET_BILLING" "$BQ_DATASET_NETWORK"; do bq rm -r -f -d "${PROJECT_ID}:${DS}"; done
for AT in telco-governance telco-semantics; do gcloud dataplex aspect-types delete "$AT" --location=eu --project="$PROJECT_ID" --quiet; done
for SA in "$PERSONA_ADMIN_SA" "$PERSONA_SPECIALIST_SA"; do
  gcloud projects remove-iam-policy-binding "$PROJECT_ID" --member="serviceAccount:${SA}" --role=roles/bigquery.jobUser --condition=None --format=none
  gcloud iam service-accounts delete "$SA" --project="$PROJECT_ID" --quiet
done
```
</details>

In [ ]:
# Step 11.2: Delete workshop resources in reverse dependency order (dry run unless CONFIRM_RESOURCE_CLEANUP = True)
from collections import Counter

CONFIRM_RESOURCE_CLEANUP = False   # Set to True and re-run this cell to delete; False only lists and prints

GONE = ("not_found", "not found", "cannot find", "does not exist", "matched no objects")
results = []

def report(label, status, detail=""):
    results.append((status, label))
    print(f"   -> {status}: {label}" + (f"\n      {detail.strip()[-300:]}" if status == "FAILED" else ""))

def run(label, cmd):
    """Print a CLI delete command; when confirmed, run it. A 'not found' error means the resource is already gone."""
    print(f"$ {cmd}")
    if CONFIRM_RESOURCE_CLEANUP:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        out = r.stdout + r.stderr
        report(label, "done" if r.returncode == 0 else "already gone" if any(g in out.lower() for g in GONE) else "FAILED", out)

def rest_delete(label, url):
    """Same for REST-only resources: print the equivalent curl; when confirmed, send the DELETE. Returns True if it was accepted."""
    print(f'$ curl -X DELETE "{url}" -H "Authorization: Bearer $(gcloud auth print-access-token)"')
    if not CONFIRM_RESOURCE_CLEANUP:
        return False
    r = requests.delete(url, headers=H)
    report(label, "done" if r.ok else "already gone" if r.status_code == 404 else "FAILED", f"HTTP {r.status_code} {r.text}")
    return r.ok

def remove_role(member, role, who):
    run(f"{role} of {who}", f'gcloud projects remove-iam-policy-binding {PROJECT_ID} --member="{member}" --role={role} --condition=None --format=none')

refresh_token()
engines = workshop_engines()
principals = agent_principals(engines)
print("Workshop Agent Runtime instances:", [f"{e['displayName']} ({e['name'].split('/')[-1]})" for e in engines] or "none")
print("Workshop Agent Identities       :", [p.split("/")[-1] for p in principals] or "none", "\n")

print("# 1. Lab 07: online evaluators that grade the workshop agents, then the eval results bucket")
for o in workshop_evaluators(engines):
    rest_delete(f"online evaluator {o.get('displayName', o['name'])}", f"{AR_API}/v1beta1/{o['name']}")
run("eval results bucket", f"gcloud storage rm -r gs://{PROJECT_ID}-eval")

print("\n# 2. Labs 02/06/08: Agent Runtime instances with their sessions, memories and sandboxes (force=true)")
deleting = []
for e in engines:
    if rest_delete(f"Agent Runtime instance {e['displayName']}", f"{AR_API}/v1/{e['name']}?force=true"):
        deleting.append(e["name"])
deadline = time.time() + 600
while deleting and time.time() < deadline:          # deletion is a long-running operation: wait until the instances are gone
    time.sleep(15)
    deleting = [n for n in deleting if requests.get(f"{AR_API}/v1/{n}", headers=H).status_code != 404]
if deleting:
    print("   still deleting after 10 min - continuing; check again with the audit cell")

print("\n# 3. Labs 06/03: registry entries, Cloud Run services, their images and uploaded sources; the private skill")
for svc in SERVICES:
    run(f"Agent Registry entry {svc}", f"gcloud agent-registry services delete {svc} --location={REGION} --project={PROJECT_ID} --quiet")
    run(f"Cloud Run service {svc}", f"gcloud run services delete {svc} --region={REGION} --project={PROJECT_ID} --quiet")
    run(f"container image {svc}", f"gcloud artifacts packages delete {svc} --repository=cloud-run-source-deploy --location={REGION} --project={PROJECT_ID} --quiet")
    run(f"uploaded source of {svc}", f'gcloud storage rm "gs://run-sources-{PROJECT_ID}-{REGION}/services/{svc}/**"')
for loc in SKILL_LOCATIONS:
    rest_delete(f"private skill in {loc} (with its revisions)", f"{skill_url(loc)}?force=true")

print("\n# 4. Lab 05: Model Armor templates (check MA_LOCATION and REGION) and the log-based metric")
for loc in MA_LOCATIONS:
    for tpl in MA_TEMPLATES:
        rest_delete(f"Model Armor template {tpl} ({loc})", f"{ma_api(loc)}/templates/{tpl}")
run("log-based metric pulse_blocked_prompts", f"gcloud logging metrics delete pulse_blocked_prompts --project={PROJECT_ID} --quiet")

print("\n# 5. Labs 02-08: project roles the labs granted (agents' Agent Identities, default compute service account)")
policy = iam_policy()
for p in principals:
    for role in sorted(roles_of(policy, p) & LAB_AGENT_ROLES):
        remove_role(p, role, f"agent {p.split('/')[-1]}")
    if roles_of(policy, p) - LAB_AGENT_ROLES:
        print(f"   left in place for agent {p.split('/')[-1]} (granted by agents-cli, not by the labs):", ", ".join(sorted(roles_of(policy, p) - LAB_AGENT_ROLES)))
for role in sorted(roles_of(policy, COMPUTE_SA) & COMPUTE_SA_ROLES):
    remove_role(COMPUTE_SA, role, "the default compute SA")

print("\n# 6. Lab 00: BigQuery datasets (tables, access, catalog entries), aspect types, then the persona service accounts")
for ds in DATASETS:
    run(f"BigQuery dataset {ds}", f"bq rm -r -f -d {PROJECT_ID}:{ds}")
for aspect in ASPECT_TYPES:
    run(f"aspect type {aspect}", f"gcloud dataplex aspect-types delete {aspect} --location={KC_LOCATION} --project={PROJECT_ID} --quiet")
for sa in PERSONA_SAS:
    for role in sorted(roles_of(policy, f"serviceAccount:{sa}")):      # project binding first, then the account
        remove_role(f"serviceAccount:{sa}", role, sa.split("@")[0])
    run(f"persona service account {sa.split('@')[0]}", f"gcloud iam service-accounts delete {sa} --project={PROJECT_ID} --quiet")

if not CONFIRM_RESOURCE_CLEANUP:
    print("\nDRY RUN - nothing was deleted. Set CONFIRM_RESOURCE_CLEANUP = True and re-run this cell.")
else:
    print("\nSummary:", dict(Counter(status for status, _ in results)))
    failed = [label for status, label in results if status == "FAILED"]
    print("Failed (safe to re-run the cell):" if failed else "No failures.", ", ".join(failed))

### Step 11.3: Verify Clean State with a Post-Teardown Audit

Run the read-only *audit()* check again after completing Step 11.2. Every workshop resource category should now report *none*. The informational lines at the bottom display any residual bindings created automatically by *agents-cli* (which reference decommissioned Agent Identity principals) and any standalone Agent Gateway resources.

Equivalent terminal verification commands:
```bash
gcloud run services list --region="$REGION" --project="$PROJECT_ID"
gcloud agent-registry services list --location="$REGION" --project="$PROJECT_ID"
gcloud iam service-accounts list --project="$PROJECT_ID" --filter="email~^persona-"
gcloud projects get-iam-policy "$PROJECT_ID" --flatten="bindings[].members" --filter="bindings.members~principal://" --format="table(bindings.role,bindings.members)"
```

In [ ]:
# Step 11.3: Run read-only audit to confirm what remains in the project
audit()

### 🧪 Try It Yourself (5 Minutes Each)
1. **Trace every teardown step back to its architecture layer.** Run Step 10.2 in dry-run mode and map each printed command to the lab that provisioned it. If an engineer deleted only the Agent Runtime instance in the console and walked away, which resources in the list would continue running or accruing storage and compute charges?
2. **Verify idempotency.** After running a confirmed teardown (*CONFIRM_RESOURCE_CLEANUP = True*), execute Step 10.2 a second time. Confirm that every step reports *already gone* and zero steps report *FAILED*.
3. **Harden for CI/CD ephemeral environments.** Why does this interactive cleanup leave the CLI-managed Agent Identity roles in place, and how would you modify the IAM filter in Step 10.2 if this script were tearing down nightly integration-test environments in a shared staging project?

## 💶 Cost and Scaling

### Residual Cost Risks If You Skip Cleanup

| Leftover resource | Ongoing cost while provisioned (list price, verified 2026-10-20) |
| :--- | :--- |
| Code Execution sandbox | **≈ \$0.18/h** at default sizing (2 vCPU / 1.5 GB) until its TTL expires (up to 14 days ≈ **\$61**) |
| Agent Runtime instances | Idle instances are not billed today (Preview), though pricing terms note this is subject to change; delete unused instances |
| Artifact Registry container images | **\$0.10/GiB-month** beyond the 0.5 GiB free tier |
| Cloud Run services | **\$0** when scaled to zero. If configured with *--min-instances=1*, ≈ \$0.11 per service per 8-hour workday |
| Online evaluation monitors | Consumes LLM judge tokens on every sampled trace while traffic hits the monitored agent |
| Cloud Storage buckets and BigQuery datasets | \$0.020/GB-month (*europe-west4*) and \$0.02/GiB-month (*EU*); workshop data is only a few megabytes (≈ \$0) |
| Managed sessions, memories, and skills | \$0.30/GiB-month beyond 1 GiB free; workshop state is a few kilobytes (≈ \$0) |
| Model Armor templates and floor settings | \$0 when no prompts or responses are being screened |

**Lab 11 execution cost: \$0.00.** Running the teardown (or deleting a temporary workshop project) stops all residual consumption immediately.

### Per-Participant Workshop Cost Summary (Full Run Across *eu* and *europe-west4*)

| Lab | Primary cost driver | ≈ List price | ≈ With 50% Flash credit (through 2026-12-31) |
| :--- | :--- | :--- | :--- |
| **Lab 00** | Environment setup: BigQuery batch load, Knowledge Catalog aspects, smoke test within free tiers | \$0.00 | \$0.00 |
| **Lab 01** | ~15 multi-agent turns on *gemini-3.8-flash* (*eu*) + 1 *gemini-3.8-live* voice turn | \$1.00 | \$0.50 |
| **Lab 02** | ~5 turns via Agent Runtime (*europe-west4*) + initial Cloud Build container image build | \$0.35 | \$0.20 |
| **Lab 03** | ~9 local turns invoking managed BigQuery/Knowledge Catalog MCPs, Cloud Run MCP, and Skill Registry | \$0.60 | \$0.30 |
| **Lab 04** | ~7 local turns, Memory Bank extraction (*gemini-3.5-flash*), ~15 min Code Execution sandbox | \$0.60 | \$0.40 |
| **Lab 05** | ~5 local turns; Model Armor screening within the 2M free tokens/month tier | \$0.30 | \$0.15 |
| **Lab 06** | ~5 turns (local + remote A2A on Cloud Run) + single final v5 Agent Runtime deployment | \$0.35 | \$0.20 |
| **Lab 07** | LLM-as-a-judge evaluation runs + agent test-case generation turns | \$1.00 | \$0.55 |
| **Lab 08** | ~5 workflow turns + optional second Agent Runtime instance (*pulse-workflow-agent*) | \$0.30 | \$0.15 |
| **Labs 09 and 10** | Conceptual Gemini Enterprise architecture guide and reverse-dependency resource teardown | \$0.00 | \$0.00 |
| **Total** | **≈ 90% driven by Gemini token consumption** | **≈ \$4.50** | **≈ \$2.50 to \$2.65** |

* **Capacity planning guidance:** Expect **~\$2.65 per participant** with the 50% Gemini 3.8 Flash promotional credit active through 2026-12-31 (**~\$4.50 at list price**). Budget **~\$10 per participant** (≈ \$300 for a 30-person workshop) to give attendees comfortable headroom for extra experimentation and retries.
* **Official pricing references:** [Gemini platform pricing](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing) · [Generative AI pricing](https://cloud.google.com/gemini-enterprise-agent-platform/generative-ai/pricing) · [Cloud Run pricing](https://cloud.google.com/run/pricing) · [BigQuery pricing](https://cloud.google.com/bigquery/pricing)

## 🎓 Workshop Conclusion and Pulse Telco Architecture Recap

You have completed the **Pulse Telco: Building Enterprise AI Agents on Gemini Enterprise Agent Platform (GEAP)** workshop.

Stepping back into the architect's control room, notice how the engineering loop you practiced mirrors a real enterprise delivery lifecycle: prove the multi-agent topology and data-access boundaries locally first (Lab 01), establish the managed cloud runtime and identity anchor early (Lab 02), iterate rapidly on your local workstation against cloud-hosted MCP servers, skills, memory, sandboxes, and Model Armor guardrails without waiting for container rebuilds on every change (Labs 03 to 05), federate across team boundaries via A2A and promote the hardened release to the cloud (Lab 06), and lock in quality and operational governance with automated evaluations, typed workflows, network egress governance, and enterprise front-door publishing (Labs 07 to 10).

| Lab | What Pulse Concierge Gained | Platform Capabilities Used |
| :--- | :--- | :--- |
| **Lab 00** | Governed project foundation: three BigQuery domain datasets (*telco_crm*, *telco_billing*, *telco_network*), Knowledge Catalog governance and semantic aspects, and two IAM personas (Alex on all three domains; Elena on CRM only) | BigQuery (EU), Knowledge Catalog (eu), IAM, Agents CLI |
| **Lab 01** | Local hierarchical multi-agent ADK application supporting text chat and bidirectional live voice in a shared session, with per-persona BigQuery IAM enforcement verified in Cloud Audit Logs | ADK *Agent* + *AgentTool*, *Runner*, Gemini (*eu*), BigQuery audit logs |
| **Lab 02** | First cloud deployment: managed Agent Runtime instance in *europe-west4* provisioned with a dedicated Agent Identity, managed sessions, an A2A endpoint, and automatic Agent Registry cataloging | Agent Runtime (*europe-west4*), Agent Identity, Agent Registry |
| **Lab 03** | Decoupled tools and domain knowledge: Google-managed BigQuery and Knowledge Catalog MCP servers, a custom IAM Context MCP server on Cloud Run, and public/private skills (*eu*), tested locally against the Lab 02 runtime | Remote MCP, Cloud Run (*europe-west4*), Agent Registry, Skill Registry (*SkillToolset*) |
| **Lab 04** | Stateful personalization and computation: persistent cloud Sessions, long-term Memory Bank extraction, isolated Code Execution sandboxes, user feedback capture, and OpenTelemetry trace/log export | Sessions, Memory Bank, Code Execution, Feedback, OpenTelemetry |
| **Lab 05** | Inline runtime security: Model Armor screening across user prompts, tool outputs (*after_tool_callback*), and model responses (*eu*), adversarial red-team verification, and a Cloud Logging metric for blocked attacks | Model Armor (*ModelArmorPlugin*), Cloud Logging |
| **Lab 06** | Cross-team federation and production release: remote NOC agent on Cloud Run invoked over A2A, plus single consolidated v5 deployment to Agent Runtime verified end-to-end in the cloud | Cloud Run, A2A (*RemoteA2aAgent*), Agent Registry, Agent Runtime (*europe-west4*) |
| **Lab 07** | Continuous quality engineering: local trajectory evaluation and managed cloud evaluation with a custom governance LLM judge and side-by-side version comparison | *agents-cli eval*, Gemini platform evaluation (*europe-west4*) |
| **Lab 08** | *(Optional)* Deterministic and agentic orchestration: ADK 2.x graph workflow with concurrent fan-out, *JoinNode*, risk-based routing, and human-in-the-loop approval across chat, voice, and Pub/Sub, compared with n8n | ADK *Workflow*, *JoinNode*, *RequestInput*, Pub/Sub triggers |
| **Lab 09** | *(Self-study, architecture reference)* Out-of-process network egress governance: default-deny proxy, SPIFFE workload identity, per-tool IAM v3 UAP CEL rules (*read_only_hint == true*), and wire-level Model Armor (*CONTENT_AUTHZ*) | Agent Gateway (*AGENT_TO_ANYWHERE*), Agent Registry, IAP UAP CEL, Model Armor |
| **Lab 10** | *(Self-study, architecture reference)* Enterprise front-door integration: publishing ADK and A2A agents to Gemini Enterprise, per-user OAuth authorization resources for BigQuery and Microsoft Entra ID / M365, and channel boundaries | Gemini Enterprise, *agents-cli publish gemini*, OAuth 2.0 authorization resources |
| **Lab 11** | Production hygiene: idempotent reverse-dependency resource teardown and read-only project verification | *gcloud*, REST APIs, IAM policy inspection |